In [4]:
import numpy as np
import pandas as pd

In [5]:
import re
from collections import Counter

# Data Pre-Processing

In [6]:
df = pd.read_csv("spam.csv", encoding="latin-1")

In [7]:
df = pd.read_csv("spam.csv", encoding="latin-1", usecols=[0,1], names=["label","message"], header=0)

In [8]:
df.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
df.nunique()

label         2
message    5169
dtype: int64

In [10]:
df["label"].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

In [11]:
def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text.split()

In [12]:
tokenize("Karthik Balaji")

['karthik', 'balaji']

In [13]:
spam_words = []
ham_words = []

In [14]:
for label, msg in zip(df["label"], df["message"]):
    words = tokenize(msg)
    if label == "spam":
        spam_words.extend(words)
    else:
        ham_words.extend(words)

In [15]:
spam_counts = Counter(spam_words)
ham_counts = Counter(ham_words)

N = 600

A = set([w for w,_ in spam_counts.most_common(N)])
B = set([w for w,_ in ham_counts.most_common(N)])

spam_indicator_words = A - B
ham_indicator_words = B - A


In [16]:
print(len(spam_indicator_words))
print(len(ham_indicator_words))

386
386


In [17]:
top_N = 200

In [ ]:
top_spam = list(spam_indicator_words)[:top_N]
top_ham = list(ham_indicator_words)[:top_N]


cols_spam = {
    f"spam_{w}": df["message"].apply(lambda x: tokenize(x).count(w))
    for w in top_spam
}

cols_ham = {
    f"ham_{w}": df["message"].apply(lambda x: tokenize(x).count(w))
    for w in top_ham
}

df = pd.concat([df, pd.DataFrame(cols_spam), pd.DataFrame(cols_ham)], axis=1).copy()

In [19]:
from sklearn.utils import resample

df_major = df[df.label == "ham"]
df_minor = df[df.label == "spam"]

df_minor_up = resample(
    df_minor,
    replace=True,
    n_samples=len(df_major),
    random_state=42
)

df_balanced = pd.concat([df_major, df_minor_up]).sample(frac=1, random_state=42)


In [20]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [21]:

X = df_balanced.drop(columns=["label", "message"])
y = df_balanced["label"].map({"ham":0, "spam":1})

In [22]:

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# SVM (Library)

In [23]:

clf = SVC(kernel="linear")
clf.fit(X_train, y_train)

,C,1.0
,kernel,'linear'
,degree,3
,gamma,'scale'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [24]:

preds = clf.predict(X_test)


In [25]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

y_pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 0.9393782383419689
[[932  33]
 [ 84 881]]
              precision    recall  f1-score   support

           0       0.92      0.97      0.94       965
           1       0.96      0.91      0.94       965

    accuracy                           0.94      1930
   macro avg       0.94      0.94      0.94      1930
weighted avg       0.94      0.94      0.94      1930



In [26]:
class MultinomialNaiveBayes:
    """
    Multinomial Naive Bayes implemented with only numpy and pandas.
    Expects non-negative count features (e.g., word counts).
    """
    def __init__(self, alpha=1.0):
        self.alpha = float(alpha)
        self.class_log_prior_ = None        # log P(class)
        self.feature_log_prob_ = None       # log P(feature|class)
        self.classes_ = None
        self.class_count_ = None
        self.feature_count_ = None

    def fit(self, X, y):
        """
        Fit the model.
        X : pandas.DataFrame or numpy.ndarray (n_samples, n_features)
        y : pandas.Series, list or numpy.ndarray (n_samples,)
        """
        if isinstance(X, pd.DataFrame):
            Xmat = X.values.astype(np.float64)
        else:
            Xmat = np.asarray(X, dtype=np.float64)

        y_arr = np.asarray(y)
        classes, class_indices = np.unique(y_arr, return_inverse=True)
        n_classes = classes.shape[0]
        n_features = Xmat.shape[1]

        class_count = np.zeros(n_classes, dtype=np.float64)
        feature_count = np.zeros((n_classes, n_features), dtype=np.float64)

        for ci in range(n_classes):
            mask = class_indices == ci
            Xc = Xmat[mask]
            class_count[ci] = Xc.shape[0]
            if Xc.size:
                feature_count[ci, :] = Xc.sum(axis=0)

        # Store counts
        self.classes_ = classes
        self.class_count_ = class_count
        self.feature_count_ = feature_count

        # class log prior: log(N_c / N)
        total_count = class_count.sum()
        self.class_log_prior_ = np.log(class_count) - np.log(total_count)

        # feature log prob with Laplace smoothing
        smoothed_fc = feature_count + self.alpha
        smoothed_cc = smoothed_fc.sum(axis=1).reshape(-1, 1)  # sum per class
        self.feature_log_prob_ = np.log(smoothed_fc) - np.log(smoothed_cc)

        return self

    def _joint_log_likelihood(self, X):
        """
        Compute unnormalized log-probability of X for each class:
        log P(class) + sum_j x_j * log P(feature_j | class)
        X : numpy array (n_samples, n_features)
        returns (n_samples, n_classes)
        """
        if isinstance(X, pd.DataFrame):
            Xmat = X.values.astype(np.float64)
        else:
            Xmat = np.asarray(X, dtype=np.float64)

        # shape: (n_samples, n_features) dot (n_features, n_classes) -> (n_samples, n_classes)
        # but feature_log_prob_ is (n_classes, n_features) so transpose
        jll = Xmat.dot(self.feature_log_prob_.T) + self.class_log_prior_
        return jll

    def predict_log_proba(self, X):
        jll = self._joint_log_likelihood(X)
        # normalize to log probabilities per sample
        # logsumexp for numerical stability
        a_max = np.max(jll, axis=1, keepdims=True)
        exp = np.exp(jll - a_max)
        sum_exp = exp.sum(axis=1, keepdims=True)
        log_proba = jll - a_max - np.log(sum_exp) + a_max - a_max  # simplifies to jll - logsumexp
        # simpler: compute log_proba = jll - logsumexp(jll)
        logsumexp = np.log(exp.sum(axis=1, keepdims=True)) + a_max
        log_proba = jll - logsumexp
        return log_proba  # shape (n_samples, n_classes)

    def predict_proba(self, X):
        return np.exp(self.predict_log_proba(X))

    def predict(self, X):
        jll = self._joint_log_likelihood(X)
        indices = np.argmax(jll, axis=1)
        return self.classes_[indices]

    def score(self, X, y):
        y_pred = self.predict(X)
        y_true = np.asarray(y)
        return np.mean(y_pred == y_true)

    def confusion_matrix(self, X, y):
        preds = self.predict(X)
        y_true = np.asarray(y)
        classes = self.classes_
        cm = np.zeros((classes.size, classes.size), dtype=int)
        class_to_index = {c:i for i,c in enumerate(classes)}
        for t, p in zip(y_true, preds):
            cm[class_to_index[t], class_to_index[p]] += 1
        return pd.DataFrame(cm, index=classes, columns=classes)

In [27]:
mnb = MultinomialNaiveBayes(alpha=1.0)
mnb.fit(X_train, y_train)
print("Accuracy:", mnb.score(X_test, y_test))
print(mnb.confusion_matrix(X_test, y_test))

Accuracy: 0.9305699481865285
     0    1
0  943   22
1  112  853


In [29]:
class KNN():
    def __init__(self):
        self.X_train = None
        self.y_train = None
        
    def fit(self, X, y, k=3):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        self.k = k

    def euclidean(self, a, b):
        return np.sqrt(np.sum((a - b) ** 2, axis=1))

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = []
        for x in X:
            dists = self.euclidean(self.X_train, x)
            indices = np.argsort(dists)
            top_k = indices[:self.k]
            labels = self.y_train[top_k]
            values, counts = np.unique(labels, return_counts=True)
            preds.append(values[np.argmax(counts)])
        return np.array(preds)


In [30]:
KNN_Model = KNN()

In [31]:
KNN_Model.fit(X_train, y_train)
y_pred = KNN_Model.predict(X_test)

In [32]:
y_pred

array([1, 0, 1, ..., 0, 1, 1], shape=(1930,))

In [36]:
np.count_nonzero(y_pred)

np.int64(928)

In [44]:
def Classification_Report(y_pred, y_test):
    y_pred = np.asarray(y_pred)
    y_test = np.asarray(y_test)

    tp = np.sum((y_pred == 1) & (y_test == 1))
    tn = np.sum((y_pred == 0) & (y_test == 0))
    fp = np.sum((y_pred == 1) & (y_test == 0))
    fn = np.sum((y_pred == 0) & (y_test == 1))

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    recall = tp / (tp + fn) if (tp + fn) != 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) != 0 else 0

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    }

In [45]:
Classification_Report(y_pred, y_test)

{'TP': np.int64(905),
 'TN': np.int64(942),
 'FP': np.int64(23),
 'FN': np.int64(60),
 'Accuracy': np.float64(0.9569948186528497),
 'Precision': np.float64(0.9752155172413793),
 'Recall': np.float64(0.9378238341968912),
 'F1': np.float64(0.9561542525092447)}

In [46]:
class NaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.class_priors = None
        self.feature_probs = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)

        classes = np.unique(y)
        self.class_priors = {}
        self.feature_probs = {}

        for c in classes:
            X_c = X[y == c]
            self.class_priors[c] = (len(X_c) + self.alpha) / (len(X) + 2 * self.alpha)

            # Bernoulli NB with Laplace smoothing
            self.feature_probs[c] = (np.sum(X_c, axis=0) + self.alpha) / (len(X_c) + 2 * self.alpha)

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = []

        for x in X:
            scores = {}
            for c in self.class_priors:
                # log space
                log_prior = np.log(self.class_priors[c])
                log_likelihood = np.sum(
                    x * np.log(self.feature_probs[c]) +
                    (1 - x) * np.log(1 - self.feature_probs[c])
                )
                scores[c] = log_prior + log_likelihood
            preds.append(max(scores, key=scores.get))

        return np.array(preds)